# Tratamento — IBGE (Bronze -> Silver)

Lê os três Parquet da camada Bronze ([extracao_bronze_ibge.ipynb](extracao_bronze_ibge.ipynb)) — `escolaridade`, `populacao`, `renda` — e aplica as transformações necessárias para a Silver, mantendo as três tabelas **separadas** (schemas e unidades diferentes, como visto em [exploracao_bronze_ibge.ipynb](exploracao_bronze_ibge.ipynb)):

1. **Deduplicação** — proteção estrutural contra reprocessamento futuro (mesmo raciocínio da Silver do ISAPS).
2. **Separação de metadados** — `tabela_codigo`/`tabela_titulo`/`variavel` saem das linhas de fato e viram uma tabela de metadados à parte, referenciada por `tabela_codigo`.
3. **Extração de unidade** — `Mil pessoas`/`Reais` saem do texto de `variavel` e entram no **nome da coluna de valor**.
4. **Renomeação da coluna de valor** — `valor` -> `quantidade` em `escolaridade`/`populacao` (é uma contagem); `renda` mantém o nome `valor` (é uma média, ver etapa 5).
5. **Anualização da `renda`** — a série trimestral vira anual (média dos trimestres disponíveis), documentando a lacuna da pandemia em vez de inventar dado.
6. **Validação de schema (`pandera`)**, uma por tabela, ao final — o contrato final da Silver, incluindo a documentação explícita de que o valor de `renda` é uma média.

## 1. Imports e configuração

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from minio import Minio
from pandera.pandas import Column, Check, DataFrameSchema

In [2]:
load_dotenv(Path.cwd().parent / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BRONZE_PREFIX = "dados_IBGE/"

BUCKET_SILVER = os.getenv("BUCKET_SILVER", "silver")
SILVER_PREFIX = "dados_IBGE/"

SILVER_DIR = Path.cwd().parent / "dados_processados" / "silver" / "dados_IBGE"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(BUCKET_SILVER):
    client.make_bucket(BUCKET_SILVER)
    print(f"Bucket '{BUCKET_SILVER}' criado.")

print(f"MinIO: {MINIO_ENDPOINT} | bucket bronze: {BUCKET_BRONZE} | bucket silver: {BUCKET_SILVER}")
print(f"Saida parquet local: {SILVER_DIR}")

MinIO: localhost:9000 | bucket bronze: bronze | bucket silver: silver
Saida parquet local: C:\Projeto_AI\dados_processados\silver\dados_IBGE


## 2. Carregamento da camada Bronze

Lê os três Parquet direto do MinIO (bucket `bronze`, prefixo `dados_IBGE/`) para três DataFrames separados — mesma decisão de manter as tabelas distintas já tomada na EDA, porque cada uma representa uma entidade e uma unidade diferentes.

In [3]:
def ler_parquet_bronze(nome_arquivo):
    data = client.get_object(BUCKET_BRONZE, f"{BRONZE_PREFIX}{nome_arquivo}").read()
    return pd.read_parquet(pd.io.common.BytesIO(data))


esc = ler_parquet_bronze("escolaridade.parquet")
pop = ler_parquet_bronze("populacao.parquet")
ren = ler_parquet_bronze("renda.parquet")

for nome, df in [("escolaridade", esc), ("populacao", pop), ("renda", ren)]:
    print(f"{nome}: {df.shape}")

escolaridade: (98, 10)
populacao: (14, 10)
renda: (72, 9)


## 3. Etapa 1 — Deduplicação

Remove linhas totalmente repetidas em cada uma das três tabelas. A EDA não encontrou duplicatas nos dados atuais — a etapa aqui cumpre a mesma função de proteção estrutural já explicada na Silver do ISAPS ([tratamento_silver_isaps.ipynb](tratamento_silver_isaps.ipynb), seção 3): custo zero hoje, proteção contra um reprocessamento futuro (por exemplo, se a extração Bronze for rodada de novo sobre o mesmo CSV e o resultado for concatenado por engano) que poderia inflar silenciosamente somas e médias.

In [4]:
for nome, df in [("escolaridade", esc), ("populacao", pop), ("renda", ren)]:
    antes = len(df)
    df.drop_duplicates(inplace=True)
    print(f"{nome}: antes={antes} | depois={len(df)} | removidas={antes - len(df)}")

escolaridade: antes=98 | depois=98 | removidas=0
populacao: antes=14 | depois=14 | removidas=0
renda: antes=72 | depois=72 | removidas=0


## 4. Etapa 2 — Extração da unidade e separação dos metadados

`tabela_codigo`, `tabela_titulo` e `variavel` são idênticos em todas as linhas de uma mesma tabela — são metadados da tabela inteira, não do registro individual (achado da EDA, seção 10). Aqui eles são retirados das linhas de fato e agrupados numa tabela `metadados` à parte, com uma linha por tabela de origem; cada linha de fato mantém só `tabela_codigo` como referência (chave estrangeira), em vez de repetir o título e a descrição da variável em cada uma das 98/14/72 linhas.

In [5]:
def extrair_unidade(variavel: str) -> str:
    m = re.search(r"\(([^()]+)\)\s*$", variavel)
    if not m:
        raise ValueError(f"Nao foi possivel extrair unidade de: {variavel!r}")
    return m.group(1)


def slug_unidade(unidade: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", unidade.lower()).strip("_")


TABELAS = {
    "escolaridade": {"df": esc, "prefixo_coluna": "quantidade", "tipo_valor": "contagem"},
    "populacao": {"df": pop, "prefixo_coluna": "quantidade", "tipo_valor": "contagem"},
    "renda": {"df": ren, "prefixo_coluna": "valor", "tipo_valor": "media"},
}

linhas_metadados = []
for nome, cfg in TABELAS.items():
    df = cfg["df"]
    tabela_codigo = df["tabela_codigo"].iloc[0]
    tabela_titulo = df["tabela_titulo"].iloc[0]
    variavel = df["variavel"].iloc[0]
    unidade = extrair_unidade(variavel)

    cfg["unidade"] = unidade
    cfg["coluna_valor"] = f"{cfg['prefixo_coluna']}_{slug_unidade(unidade)}"

    linhas_metadados.append({
        "tabela_codigo": tabela_codigo,
        "tabela_titulo": tabela_titulo,
        "variavel": variavel,
        "unidade": unidade,
        "tipo_valor": cfg["tipo_valor"],
    })

    cfg["df"] = df.drop(columns=["tabela_titulo", "variavel"])

metadados = pd.DataFrame(linhas_metadados)
metadados

,tabela_codigo,tabela_titulo,variavel,unidade,tipo_valor
0,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Mil pessoas,contagem
1,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Mil pessoas,contagem
2,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Reais,media


`escolaridade` e `populacao` já são anuais desde a Bronze — `trimestre` é nulo em 100% das linhas (a pergunta "qual trimestre" não se aplica) e `periodo_original` é só a mesma informação de `ano` em formato texto. Para que **todas** as tabelas da Silver fiquem "em função do ano" de forma uniforme (sem colunas trimestrais sobrando onde não fazem sentido, e sem uma coluna duplicando outra), essas duas colunas são removidas aqui — `renda` perde as equivalentes de forma natural na anualização da etapa 6, então esta etapa deixa as três tabelas consistentes entre si desde já.

In [6]:
for nome, cfg in TABELAS.items():
    if nome == "renda":
        continue  # renda ainda esta trimestral aqui; perde periodo_original/trimestre na anualizacao (etapa 6)
    cfg["df"] = cfg["df"].drop(columns=["periodo_original", "trimestre"])

esc, pop = TABELAS["escolaridade"]["df"], TABELAS["populacao"]["df"]
esc.head(3)

,tabela_codigo,regiao,ano,sexo,valor,nivel_instrucao
0,7128,Brasil,2016,Homens,4634.0,Sem instrução
1,7128,Brasil,2016,Mulheres,4913.0,Sem instrução
2,7128,Brasil,2017,Homens,4249.0,Sem instrução


## 5. Etapa 3 — Renomeação da coluna de valor

`valor` vira `quantidade_<unidade>` em `escolaridade`/`populacao` (é uma contagem de pessoas — soma entre categorias tem significado, como usado na verificação cruzada da EDA) e `valor_<unidade>` em `renda` (continua se chamando `valor`, não `quantidade`, exatamente para deixar visualmente marcado que **não é uma contagem somável** — é uma média mensal, e o nome da coluna sozinho já é um lembrete disso, reforçado pela documentação de schema na etapa final).

In [7]:
for nome, cfg in TABELAS.items():
    cfg["df"] = cfg["df"].rename(columns={"valor": cfg["coluna_valor"]})
    print(f"{nome}: coluna de valor -> '{cfg['coluna_valor']}'")

esc, pop, ren = TABELAS["escolaridade"]["df"], TABELAS["populacao"]["df"], TABELAS["renda"]["df"]
esc.head(3)

escolaridade: coluna de valor -> 'quantidade_mil_pessoas'
populacao: coluna de valor -> 'quantidade_mil_pessoas'
renda: coluna de valor -> 'valor_reais'


,tabela_codigo,regiao,ano,sexo,quantidade_mil_pessoas,nivel_instrucao
0,7128,Brasil,2016,Homens,4634.0,Sem instrução
1,7128,Brasil,2016,Mulheres,4913.0,Sem instrução
2,7128,Brasil,2017,Homens,4249.0,Sem instrução


## 6. Etapa 4 — Anualização da `renda`

`escolaridade` e `populacao` já são séries anuais. `renda` é trimestral e precisa virar anual — a agregação usada é a **média dos trimestres disponíveis** em cada ano (nunca soma: o valor já é, ele mesmo, uma média mensal; a "soma de médias trimestrais" não teria significado nenhum).

**Sobre a lacuna da pandemia:** a série (PNAS Contínua) tem cobertura trimestral incompleta em dois pontos — `2020` só tem o 1º trimestre, e `2021` não tem nenhum trimestre — porque a pesquisa foi suspensa/não divulgada nesse período da pandemia de covid-19 (achado documentado na EDA, seção 7). A regra desta etapa é: **não inventar** os trimestres/o ano ausente para completar a série. Isso tem duas consequências, deixadas explícitas nos dados em vez de escondidas:

- **`2021` simplesmente não aparece como linha** na Silver — não existe dado real para preencher, e gerar uma linha com valor interpolado/zerado passaria a informação falsa de que a pesquisa mediu alguma coisa naquele ano.
- **`2020` aparece, mas com uma coluna `qtd_trimestres_disponiveis` = 1** (contra 4 dos anos completos) — a linha é real (vem de um trimestre efetivamente pesquisado), mas sinalizada como uma média bem menos robusta que a dos demais anos, para quem for consumir a Silver não tratar `2020` com a mesma confiança dos outros anos sem essa ressalva.

In [8]:
ren_anual = (
    ren.groupby(["tabela_codigo", "regiao", "ano", "sexo"], as_index=False)
    .agg(
        valor_reais=("valor_reais", "mean"),
        qtd_trimestres_disponiveis=("valor_reais", "count"),
    )
)
ren_anual["valor_reais"] = ren_anual["valor_reais"].round(2)

TABELAS["renda"]["df"] = ren_anual
ren = ren_anual

print(f"Anos em renda apos anualizacao: {sorted(ren['ano'].unique())}")
print("(2021 fica de fora por nao ter nenhum trimestre pesquisado - ver explicacao acima)")
ren.sort_values(["ano", "sexo"])

Anos em renda apos anualizacao: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2022), np.int64(2023), np.int64(2024)]
(2021 fica de fora por nao ter nenhum trimestre pesquisado - ver explicacao acima)


,tabela_codigo,regiao,ano,sexo,valor_reais,qtd_trimestres_disponiveis
0,5436,Brasil,2014,Homens,3765.75,4
1,5436,Brasil,2014,Mulheres,2801.00,4
2,5436,Brasil,2015,Homens,3713.00,4
3,5436,Brasil,2015,Mulheres,2830.50,4
4,5436,Brasil,2016,Homens,3680.25,4
5,5436,Brasil,2016,Mulheres,2882.50,4
6,5436,Brasil,2017,Homens,3745.00,4
7,5436,Brasil,2017,Mulheres,2913.00,4
8,5436,Brasil,2018,Homens,3803.50,4
9,5436,Brasil,2018,Mulheres,2950.25,4


## 7. Etapa 5 — Validação de schema (`pandera`), uma por tabela

Cada tabela de fato ganha seu próprio `DataFrameSchema` — os domínios são diferentes (`nivel_instrucao` só existe em `escolaridade`, `qtd_trimestres_disponiveis` só em `renda`), então um schema único e genérico esconderia justamente as diferenças que esta Silver preservou de propósito. Roda por último, como de costume: prova automaticamente que as etapas anteriores produziram o contrato de dado esperado, em vez de confiar visualmente na prévia das células anteriores.

A distinção **contagem x média** discutida nas etapas 3-4 fica documentada tanto aqui, no `description` da coluna de valor de cada schema (visível a quem inspecionar o schema programaticamente), quanto na tabela `metadados` (coluna `tipo_valor`).

In [9]:
schema_metadados = DataFrameSchema(
    {
        "tabela_codigo": Column(str, nullable=False, unique=True),
        "tabela_titulo": Column(str, nullable=False),
        "variavel": Column(str, nullable=False),
        "unidade": Column(str, nullable=False),
        "tipo_valor": Column(str, Check.isin(["contagem", "media"]), nullable=False),
    },
    coerce=True,
    strict=True,
)

schema_escolaridade = DataFrameSchema(
    {
        "tabela_codigo": Column(str, nullable=False),
        "regiao": Column(str, nullable=False),
        "ano": Column(int, Check.in_range(2000, 2100), nullable=False),
        "sexo": Column(str, Check.isin(["Homens", "Mulheres"]), nullable=False),
        "nivel_instrucao": Column(str, nullable=False),
        "quantidade_mil_pessoas": Column(
            float, Check.ge(0), nullable=False,
            description="Contagem de pessoas (milhares). Pode ser somada entre categorias de nivel_instrucao.",
        ),
    },
    coerce=True,
    strict=True,
)

schema_populacao = DataFrameSchema(
    {
        "tabela_codigo": Column(str, nullable=False),
        "regiao": Column(str, nullable=False),
        "ano": Column(int, Check.in_range(2000, 2100), nullable=False),
        "sexo": Column(str, Check.isin(["Homens", "Mulheres"]), nullable=False),
        "grupo_idade": Column(str, nullable=False),
        "quantidade_mil_pessoas": Column(
            float, Check.ge(0), nullable=False,
            description="Contagem de pessoas (milhares). Pode ser somada entre categorias.",
        ),
    },
    coerce=True,
    strict=True,
)

schema_renda = DataFrameSchema(
    {
        "tabela_codigo": Column(str, nullable=False),
        "regiao": Column(str, nullable=False),
        "ano": Column(int, Check.in_range(2000, 2100), nullable=False),
        "sexo": Column(str, Check.isin(["Homens", "Mulheres"]), nullable=False),
        "qtd_trimestres_disponiveis": Column(int, Check.in_range(1, 4), nullable=False),
        "valor_reais": Column(
            float, Check.ge(0), nullable=False,
            description=(
                "MEDIA do rendimento mensal (reais), ja anualizada a partir dos trimestres "
                "disponiveis (ver qtd_trimestres_disponiveis). NAO e uma contagem: nao somar "
                "entre anos/sexo. 2021 nao aparece nesta tabela - a pesquisa nao foi divulgada "
                "nesse periodo da pandemia de covid-19, e nenhum valor foi inventado para preencher a lacuna."
            ),
        ),
    },
    coerce=True,
    strict=True,
)

metadados = schema_metadados.validate(metadados)
esc = schema_escolaridade.validate(esc)
pop = schema_populacao.validate(pop)
ren = schema_renda.validate(ren)
print("Todos os schemas validos.")

Todos os schemas validos.


## 8. Gravação em Parquet (camada Silver)

Quatro arquivos — um por tabela de fato mais a tabela de metadados — gravados localmente em `dados_processados/silver/dados_IBGE/` e enviados ao MinIO no bucket `silver`, prefixo `dados_IBGE/`.

In [10]:
tabelas_finais = {
    "metadados": metadados,
    "escolaridade": esc,
    "populacao": pop,
    "renda": ren,
}

arquivos_gerados = {}
for nome, df in tabelas_finais.items():
    out_path = SILVER_DIR / f"{nome}.parquet"
    df.to_parquet(out_path, engine="pyarrow", index=False)
    arquivos_gerados[nome] = out_path

    object_name = f"{SILVER_PREFIX}{nome}.parquet"
    client.fput_object(BUCKET_SILVER, object_name, str(out_path))
    print(f"Gravado: {out_path} ({len(df)} linhas) -> s3://{BUCKET_SILVER}/{object_name}")

Gravado: C:\Projeto_AI\dados_processados\silver\dados_IBGE\metadados.parquet (3 linhas) -> s3://silver/dados_IBGE/metadados.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\dados_IBGE\escolaridade.parquet (98 linhas) -> s3://silver/dados_IBGE/escolaridade.parquet
Gravado: C:\Projeto_AI\dados_processados\silver\dados_IBGE\populacao.parquet (14 linhas) -> s3://silver/dados_IBGE/populacao.parquet


Gravado: C:\Projeto_AI\dados_processados\silver\dados_IBGE\renda.parquet (20 linhas) -> s3://silver/dados_IBGE/renda.parquet


## 9. Conferência final

In [11]:
for nome, out_path in arquivos_gerados.items():
    df = pd.read_parquet(out_path)
    print(f"{nome}: {df.shape} | colunas: {list(df.columns)}")

metadados: (3, 5) | colunas: ['tabela_codigo', 'tabela_titulo', 'variavel', 'unidade', 'tipo_valor']
escolaridade: (98, 6) | colunas: ['tabela_codigo', 'regiao', 'ano', 'sexo', 'quantidade_mil_pessoas', 'nivel_instrucao']
populacao: (14, 6) | colunas: ['tabela_codigo', 'regiao', 'ano', 'sexo', 'quantidade_mil_pessoas', 'grupo_idade']
renda: (20, 6) | colunas: ['tabela_codigo', 'regiao', 'ano', 'sexo', 'valor_reais', 'qtd_trimestres_disponiveis']


In [12]:
pd.read_parquet(arquivos_gerados['metadados'])

,tabela_codigo,tabela_titulo,variavel,unidade,tipo_valor
0,7128,"Pessoas de 14 anos ou mais de idade, por sexo ...",Pessoas de 14 anos ou mais de idade (Mil pessoas),Mil pessoas,contagem
1,7109,"População residente, por sexo e grupo de idade",População (Mil pessoas),Mil pessoas,contagem
2,5436,Rendimento médio mensal real das pessoas de 14...,Rendimento médio mensal real das pessoas de 14...,Reais,media


In [13]:
pd.read_parquet(arquivos_gerados['renda']).sort_values(['ano', 'sexo'])

,tabela_codigo,regiao,ano,sexo,valor_reais,qtd_trimestres_disponiveis
0,5436,Brasil,2014,Homens,3765.75,4
1,5436,Brasil,2014,Mulheres,2801.00,4
2,5436,Brasil,2015,Homens,3713.00,4
3,5436,Brasil,2015,Mulheres,2830.50,4
4,5436,Brasil,2016,Homens,3680.25,4
5,5436,Brasil,2016,Mulheres,2882.50,4
6,5436,Brasil,2017,Homens,3745.00,4
7,5436,Brasil,2017,Mulheres,2913.00,4
8,5436,Brasil,2018,Homens,3803.50,4
9,5436,Brasil,2018,Mulheres,2950.25,4


## 10. Conclusões

- **Deduplicação:** nenhuma duplicata encontrada — segue como proteção estrutural.
- **Metadados:** `tabela_codigo`, `tabela_titulo` e `variavel` saíram das linhas de fato e viraram a tabela `metadados` (uma linha por tabela de origem); as linhas de fato mantêm só `tabela_codigo` como referência.
- **Unidade explícita:** `Mil pessoas`/`Reais` extraídos de `variavel` e incorporados ao nome da coluna de valor (`quantidade_mil_pessoas`, `valor_reais`).
- **Contagem x média:** `escolaridade`/`populacao` usam `quantidade_*` (contagem, somável); `renda` usa `valor_*` (média, não somável) — a distinção fica no nome da coluna, no `description` do schema e na coluna `tipo_valor` de `metadados`.
- **Anualização da renda:** de trimestral para anual via média dos trimestres disponíveis, com `qtd_trimestres_disponiveis` documentando quando o ano é uma média incompleta (`2020`, com só 1 trimestre). `2021` não aparece — nenhum valor foi inventado para a lacuna da pandemia.
- **Schema:** validado com sucesso, uma definição por tabela, refletindo os domínios reais de cada uma em vez de um contrato único genérico.